In [ ]:
# Portable data paths; execute notebooks from the package root.
import os
from pathlib import Path
_review_base = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
DATA_ROOT = Path(os.environ.get("REVIEW_DATA_ROOT", str(_review_base / "data"))).resolve()
RESULTS_ROOT = Path(os.environ.get("REVIEW_RESULTS_ROOT", str(_review_base / "results"))).resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


# Figure analysis and plotting

# Figure 1: Prepare annual provincial curtailment

In [ ]:
import os
import re
from pathlib import Path
 
import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
 

SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
OUTPUT_FILE = str(DATA_ROOT / '7_Figure/Data_for_figure/province_annual_curtailment.xlsx')
 

PROVINCE_ORDER = [
    ("beijing", '\u5317\u4eac'), ("tianjin", '\u5929\u6d25'), ("hebei", '\u6cb3\u5317'), ("shanxi", '\u5c71\u897f'),
    ("neimeng", '\u5185\u8499\u53e4'), ("liaoning", '\u8fbd\u5b81'), ("jilin", '\u5409\u6797'), ("heilongjiang", '\u9ed1\u9f99\u6c5f'),
    ("shanghai", '\u4e0a\u6d77'), ("jiangsu", '\u6c5f\u82cf'), ("zhejiang", '\u6d59\u6c5f'), ("anhui", '\u5b89\u5fbd'),
    ("fujian", '\u798f\u5efa'), ("jiangxi", '\u6c5f\u897f'), ("shandong", '\u5c71\u4e1c'), ("henan", '\u6cb3\u5357'),
    ("hubei", '\u6e56\u5317'), ("hunan", '\u6e56\u5357'), ("guangdong", '\u5e7f\u4e1c'), ("guangxi", '\u5e7f\u897f'),
    ("hainan", '\u6d77\u5357'), ("chongqing", '\u91cd\u5e86'), ("sichuan", '\u56db\u5ddd'), ("guizhou", '\u8d35\u5dde'),
    ("yunnan", '\u4e91\u5357'), ("xizang", '\u897f\u85cf'), ("shannxi", '\u9655\u897f'), ("gansu", '\u7518\u8083'),
    ("qinghai", '\u9752\u6d77'), ("ningxia", '\u5b81\u590f'), ("xinjiang", '\u65b0\u7586'),
]
CN_NAME = dict(PROVINCE_ORDER)
 
YEARS = [2023, 2024, 2025]
 
FILE_SUFFIX = "_total_hourly_curtailment.xlsx"
 
 
def extract_province_year(filename: str):
    'Extract province year.'
    if not filename.endswith(FILE_SUFFIX):
        return None, None
    name = filename[: -len(FILE_SUFFIX)]
    parts = name.split("_")
    for i, p in enumerate(parts):
        if re.fullmatch(r"20\d{2}", p):
            province = "_".join(parts[:i])
            return province, int(p)
    return None, None
 
 
def scan_source_dir(source_dir: str) -> pd.DataFrame:
    'Scan source dir.'
    records = []
    files = [f for f in os.listdir(source_dir) if f.endswith(FILE_SUFFIX)]
    print(f"Found {len(files)} input files in {source_dir}.")
 
    for fname in sorted(files):
        prov, year = extract_province_year(fname)
        if prov is None:
            print(f"  [Skip] Could not parse filename: {fname}")
            continue
 
        fp = os.path.join(source_dir, fname)
        df = pd.read_excel(fp)
 
        if "total_curtailment_MWh" not in df.columns:
            print(f"  [Skip] {fname}: Missing total_curtailment_MWh column")
            continue
 
        total_mwh = df["total_curtailment_MWh"].sum()
        solar_mwh = df["solar_curtailment_MWh"].sum() if "solar_curtailment_MWh" in df.columns else None
        wind_mwh = df["wind_curtailment_MWh"].sum() if "wind_curtailment_MWh" in df.columns else None
 
        if solar_mwh is not None and wind_mwh is not None:
            has_solar = solar_mwh > 0
            has_wind = wind_mwh > 0
            if has_solar and has_wind:
                source = "both"
            elif has_solar:
                source = "solar only"
            elif has_wind:
                source = "wind only"
            else:
                source = "none"
        else:
            source = "unknown"
 
        total_gwh = total_mwh / 1000.0
        records.append(
            {"province": prov, "year": year, "total_curtailment_GWh": total_gwh, "source": source}
        )
        print(f"  {prov}_{year}: [{source}] Total = {total_gwh:.2f} GWh")
 
    return pd.DataFrame(records)
 
 
def build_summary_workbook(data_df: pd.DataFrame, output_file: str):
    'Build summary workbook.'
    data = {
        (row.province, row.year): (row.total_curtailment_GWh, row.source)
        for row in data_df.itertuples()
    }
 
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Annual curtailment by province'
 
    FONT_NAME = "Arial"
    header_font = Font(name=FONT_NAME, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    title_font = Font(name=FONT_NAME, bold=True, size=14)
    normal_font = Font(name=FONT_NAME)
    partial_font = Font(name=FONT_NAME, color="B45F06")  
    bold_font = Font(name=FONT_NAME, bold=True)
    na_font = Font(name=FONT_NAME, color="999999", italic=True)
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", vertical="center")
    right = Alignment(horizontal="right", vertical="center")
 
    ws.merge_cells("A1:F1")
    ws["A1"] = 'Annual provincial wind and solar curtailment (GWh)'
    ws["A1"].font = title_font
    ws["A1"].alignment = center
 
    ws["A2"] = 'Source: hourly province-year files. Hourly curtailment (MWh) is summed and converted to GWh.'
    ws["A2"].font = Font(name=FONT_NAME, italic=True, size=9, color="666666")
    ws.merge_cells("A2:F2")
 
    header_row = 4
    headers = ['Province key', 'Province (source name)'] + [str(y) for y in YEARS] + ['Three-year total (GWh)']
    for col, h in enumerate(headers, start=1):
        c = ws.cell(row=header_row, column=col, value=h)
        c.font = header_font
        c.fill = header_fill
        c.alignment = center
        c.border = border
 
    
    known_provinces = {p for p, _ in PROVINCE_ORDER}
    extra_provinces = sorted({p for p in data_df["province"].unique() if p not in known_provinces})
    all_provinces = PROVINCE_ORDER + [(p, CN_NAME.get(p, "")) for p in extra_provinces]
 
    start_row = header_row + 1
    for i, (prov_en, prov_cn) in enumerate(all_provinces):
        r = start_row + i
        ws.cell(row=r, column=1, value=prov_en).font = normal_font
        ws.cell(row=r, column=2, value=prov_cn).font = normal_font
        ws.cell(row=r, column=1).border = border
        ws.cell(row=r, column=2).border = border
 
        for j, year in enumerate(YEARS):
            col = 3 + j
            cell = ws.cell(row=r, column=col)
            if (prov_en, year) in data:
                total, source = data[(prov_en, year)]
                cell.value = round(total, 2)
                cell.number_format = "#,##0.00"
                cell.font = normal_font if source == "both" else partial_font
            else:
                cell.value = 'No data'
                cell.font = na_font
            cell.alignment = right
            cell.border = border
 
        total_cell = ws.cell(row=r, column=6)
        total_cell.value = f'=IF(COUNT(C{r}:E{r})=0,"No data",SUM(C{r}:E{r}))'
        total_cell.number_format = "#,##0.00"
        total_cell.font = bold_font
        total_cell.alignment = right
        total_cell.border = border
 
    end_row = start_row + len(all_provinces) - 1
 
    gt_row = end_row + 1
    ws.cell(row=gt_row, column=2, value='National total').font = bold_font
    ws.cell(row=gt_row, column=2).alignment = right
    ws.cell(row=gt_row, column=1).border = border
    ws.cell(row=gt_row, column=2).border = border
    for j, year in enumerate(YEARS):
        col = 3 + j
        letter = get_column_letter(col)
        c = ws.cell(row=gt_row, column=col)
        c.value = f"=SUM({letter}{start_row}:{letter}{end_row})"
        c.number_format = "#,##0.00"
        c.font = bold_font
        c.alignment = right
        c.border = border
    c = ws.cell(row=gt_row, column=6)
    c.value = f"=SUM(F{start_row}:F{end_row})"
    c.number_format = "#,##0.00"
    c.font = bold_font
    c.alignment = right
    c.border = border
 
    note_row = gt_row + 2
    ws.cell(row=note_row, column=1, value='Notes:').font = bold_font
    notes = [
        '1. "No data" means the hourly input file is missing for that province and year.',
        '2. Orange values indicate that only solar or only wind data are available. Totals may be understated.',
        '3. Black values indicate that both solar and wind data are available.',
    ]
    for i, note in enumerate(notes):
        ws.cell(row=note_row + 1 + i, column=1, value=note).font = Font(
            name=FONT_NAME, size=9, color="444444"
        )
        ws.merge_cells(start_row=note_row + 1 + i, start_column=1, end_row=note_row + 1 + i, end_column=6)
 
    widths = [14, 12, 12, 12, 12, 16]
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w
 
    ws.freeze_panes = f"C{start_row}"
 
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_file)
    print(f"\n[OK] Summary saved: {output_file}")
 
 
def main():
    data_df = scan_source_dir(SOURCE_DIR)
    if data_df.empty:
        print('See the preceding output for processing details.')
        return
    build_summary_workbook(data_df, OUTPUT_FILE)
 
    
    pivot = data_df.pivot_table(
        index="province", columns="year", values="total_curtailment_GWh", aggfunc="sum"
    )
    print('See the preceding output for processing details.')
    print(pivot.round(2))
 
 
if __name__ == "__main__":
    main()

# Figure 1: Provincial curtailment map and rates

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams.update({'font.size': 20})

In [ ]:
# ---------- Paths ----------
CURTAIL_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
SHP_PATH = str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/GS(2024)0650-SHP/sheng.shp')
DATACENTER_FILE = str(DATA_ROOT / '5_Data_Center/datacenter_est.xlsx')

# Province short name (matches curtailment filenames) -> (province code, English display name)
PROVINCE_INFO = {
    'beijing': ('110000', 'Beijing'), 'tianjin': ('120000', 'Tianjin'), 'hebei': ('130000', 'Hebei'),
    'shanxi': ('140000', 'Shanxi'), 'neimeng': ('150000', 'Inner Mongolia'), 'liaoning': ('210000', 'Liaoning'),
    'jilin': ('220000', 'Jilin'), 'heilongjiang': ('230000', 'Heilongjiang'), 'shanghai': ('310000', 'Shanghai'),
    'jiangsu': ('320000', 'Jiangsu'), 'zhejiang': ('330000', 'Zhejiang'), 'anhui': ('340000', 'Anhui'),
    'fujian': ('350000', 'Fujian'), 'jiangxi': ('360000', 'Jiangxi'), 'shandong': ('370000', 'Shandong'),
    'henan': ('410000', 'Henan'), 'hubei': ('420000', 'Hubei'), 'hunan': ('430000', 'Hunan'),
    'guangdong': ('440000', 'Guangdong'), 'guangxi': ('450000', 'Guangxi'), 'hainan': ('460000', 'Hainan'),
    'chongqing': ('500000', 'Chongqing'), 'sichuan': ('510000', 'Sichuan'), 'guizhou': ('520000', 'Guizhou'),
    'yunnan': ('530000', 'Yunnan'), 'xizang': ('540000', 'Tibet'), 'shannxi': ('610000', 'Shaanxi'),
    'gansu': ('620000', 'Gansu'), 'qinghai': ('630000', 'Qinghai'), 'ningxia': ('640000', 'Ningxia'),
    'xinjiang': ('650000', 'Xinjiang'),
}
# ---------- 1. Read curtailment data, compute total curtailment (GWh) and curtailment rate ----------
rows = []
for fp in sorted(glob.glob(os.path.join(CURTAIL_DIR, '*_total_hourly_curtailment.xlsx'))):
    name = os.path.basename(fp).replace('_total_hourly_curtailment.xlsx', '')
    parts = name.split('_')
    year = next((p for p in parts if p in ('2023', '2024', '2025')), None)
    if year is None:
        continue
    province = '_'.join(parts[:parts.index(year)])
    df = pd.read_excel(fp, usecols=['total_curtailment_MWh', 'solar_generation_MW', 'wind_generation_MW'])
    total_curt = df['total_curtailment_MWh'].sum()
    total_gen = df['solar_generation_MW'].sum() + df['wind_generation_MW'].sum()
    rows.append({
        'province': province,
        'year': int(year),
        'curtailment_GWh': total_curt / 1000,
        'curtailment_rate': total_curt / total_gen if total_gen > 0 else np.nan,
    })
curt = pd.DataFrame(rows)
curt['province_code'] = curt['province'].map(lambda p: PROVINCE_INFO.get(p, (None, None))[0])
curt['province_name'] = curt['province'].map(lambda p: PROVINCE_INFO.get(p, (None, None))[1])

# ---------- 2. Panel a data: 2023-2025 total curtailment, merged onto province boundaries ----------
total_curt = curt.groupby(['province_code', 'province_name'], as_index=False)['curtailment_GWh'].sum()

gdf = gpd.read_file(SHP_PATH)
gdf = gdf.rename(columns={'sheng_code': 'province_code_shp'})  
gdf['province_code_shp'] = gdf['province_code_shp'].astype(str).str.replace(r'\.0$', '', regex=True)  # guard against int/float dtype mismatch on merge
gdf = gdf.merge(total_curt, left_on='province_code_shp', right_on='province_code', how='left')

# ---------- 3. Data center locations ----------
dc = pd.read_excel(DATACENTER_FILE)
dc = dc.dropna(subset=['LATITUDE', 'LONGITUDE'])

In [ ]:
# ---------- 4. Panel b data: Top 15 provinces by curtailment rate (ranked by 3-year average) ----------
top15 = (
    curt.groupby('province')['curtailment_rate'].mean()
    .sort_values(ascending=False).head(10).index.tolist()
)
pivot = (
    curt[curt['province'].isin(top15)]
    .pivot(index='province', columns='year', values='curtailment_rate')
    .reindex(top15)
)
labels = [PROVINCE_INFO[p][1] for p in pivot.index]

In [ ]:
"""
Two-panel figure:
  Panel A - China map (Albers Equal Area) with curtailment choropleth,
            data-center points, and a South China Sea inset.
  Panel B - Horizontal grouped bar chart of curtailment rate by year.

Key fixes vs. the original script:
  1. panel_ratio now uses the *physical* (inch) aspect ratio of ax1's
     box, not the raw figure-fraction width/height.
  2. aspect="equal" uses adjustable="datalim" instead of "box", so
     matplotlib never silently shrinks/repositions ax1 during the
     final draw.
  3. The choropleth colorbar is built on its own manually placed
     `cax`, so it can no longer steal space from ax1.
Together these keep ax1's frame pinned to its GridSpec box (same
height as ax2) and let the map fill that box without extra padding.

Requires `gdf`, `dc`, `pivot`, `labels` to already exist in scope
(same as the original script).
"""

import pyproj
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import ScalarFormatter, MaxNLocator

plt.close("all")

# ============================================================
# Basic settings
# ============================================================
CACHE_PATH = str(DATA_ROOT / 'external/BasicData/ChinaMap/cache.gpkg')
base_path = str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/GS(2024)0650-SHP')

LAND_SIMPLIFY = 0.05
FONT_SIZE = 15


ZOOM = 0.92

# China Albers Equal Area projection
china_proj_crs = (
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

# ============================================================
# Read and project spatial data
# ============================================================
province_proj = gpd.read_file(
    f"{base_path}/sheng.shp",
    engine="pyogrio"
).to_crs(china_proj_crs)

province_proj["geometry"] = province_proj.geometry.make_valid()

ten_lines_proj = gpd.read_file(
    f"{base_path}/tenlines.shp",
    engine="pyogrio"
).to_crs(china_proj_crs)

land_boundary_raw = gpd.read_file(f"{base_path}/continentline.shp")

land_boundary_raw["geometry"] = land_boundary_raw.geometry.simplify(
    tolerance=LAND_SIMPLIFY,
    preserve_topology=True
)

land_boundary_proj = land_boundary_raw.to_crs(china_proj_crs)

# Buffer layers
ten_lines_buffer1 = gpd.read_file(
    CACHE_PATH, layer="ten_lines_buffer1", engine="pyogrio"
).to_crs(china_proj_crs)

ten_lines_buffer2 = gpd.read_file(
    CACHE_PATH, layer="ten_lines_buffer2", engine="pyogrio"
).to_crs(china_proj_crs)

land_boundary_buffer1 = gpd.read_file(
    CACHE_PATH, layer="land_boundary_buffer1", engine="pyogrio"
).to_crs(china_proj_crs)

land_boundary_buffer2 = gpd.read_file(
    CACHE_PATH, layer="land_boundary_buffer2", engine="pyogrio"
).to_crs(china_proj_crs)

# Curtailment data
gdf_proj = gdf.to_crs(china_proj_crs)

# Data center points
dc_proj = gpd.GeoDataFrame(
    dc,
    geometry=gpd.points_from_xy(dc["LONGITUDE"], dc["LATITUDE"]),
    crs="EPSG:4326"
).to_crs(china_proj_crs)

# Coordinate transformer for longitude/latitude gridlines
to_proj = pyproj.Transformer.from_crs(
    "EPSG:4326", china_proj_crs, always_xy=True
)

# ============================================================
# Create figure and two panels
# ============================================================
fig = plt.figure(figsize=(28, 9))

ax1 = plt.subplot2grid((50, 160), (1, 0), rowspan=48, colspan=73)
ax2 = plt.subplot2grid((50, 160), (1, 85), rowspan=48, colspan=48)



fig.canvas.draw()
ax1_pos = ax1.get_position().frozen()
ax2_pos = ax2.get_position().frozen()

# ============================================================
# Panel A: define geographic extent
# ============================================================
main_xlim_ll = [78.5, 135]
main_ylim_ll = [15, 55]


lons = np.linspace(main_xlim_ll[0], main_xlim_ll[1], 500)
lats = np.linspace(main_ylim_ll[0], main_ylim_ll[1], 500)

lon_boundary = np.concatenate([
    lons, lons,
    np.full_like(lats, main_xlim_ll[0]),
    np.full_like(lats, main_xlim_ll[1])
])

lat_boundary = np.concatenate([
    np.full_like(lons, main_ylim_ll[0]),
    np.full_like(lons, main_ylim_ll[1]),
    lats, lats
])

xs, ys = to_proj.transform(lon_boundary, lat_boundary)

xmin0 = xs.min()
xmax0 = xs.max()
ymin0 = ys.min()
ymax0 = ys.max()


xcenter = (xmin0 + xmax0) / 2
ycenter = (ymin0 + ymax0) / 2

# ============================================================
# Zoom map while keeping panel border fixed
# ============================================================




fig_w, fig_h = fig.get_size_inches()
panel_ratio = (ax1_pos.width * fig_w) / (ax1_pos.height * fig_h)

target_height = (ymax0 - ymin0) * ZOOM
target_width = target_height * panel_ratio

main_xlim = (xcenter - target_width / 2, xcenter + target_width / 2)
main_ylim = (ycenter - target_height / 2, ycenter + target_height / 2)

# ============================================================
# Panel A: plot map layers
# ============================================================
land_boundary_buffer2.plot(
    ax=ax1, alpha=1, color="#c4cae3", edgecolor="none", zorder=1
)

land_boundary_buffer1.plot(
    ax=ax1, alpha=1, color="#aea8cd", edgecolor="none", zorder=2
)

ten_lines_buffer2.plot(
    ax=ax1, alpha=1, color="#c4cae3", edgecolor="none", zorder=1
)

ten_lines_buffer1.plot(
    ax=ax1, alpha=1, color="#aea8cd", edgecolor="none", zorder=2
)



gdf_proj.plot(
    column="curtailment_GWh",
    cmap="YlOrRd",
    linewidth=0.5,
    edgecolor="grey",
    ax=ax1,
    legend=False,
    missing_kwds={"color": "#e8e8e8"},
    zorder=3
)

land_boundary_proj.plot(ax=ax1, linewidth=1.0, color="black", zorder=4)
ten_lines_proj.plot(ax=ax1, linewidth=1.0, color="black", zorder=5)

ax1.scatter(
    dc_proj.geometry.x,
    dc_proj.geometry.y,
    s=15,
    color="blue",
    edgecolor="white",
    linewidth=0.3,
    zorder=6
)


ax1.set_xlim(main_xlim)
ax1.set_ylim(main_ylim)



ax1.set_aspect("equal", adjustable="datalim")


ax1.set_position(ax1_pos)






fig.canvas.draw()
main_xlim = ax1.get_xlim()
main_ylim = ax1.get_ylim()

# ============================================================
# Panel A: longitude and latitude gridlines
# ============================================================
lon_ticks = [80, 90, 100, 110, 120, 130]
lat_ticks = [20, 30, 40, 50]

lon_curves = {}
lat_curves = {}

# Longitude curves
for lon in lon_ticks:
    lats_grid = np.linspace(-10, 70, 1500)
    xs_grid, ys_grid = to_proj.transform(np.full_like(lats_grid, lon), lats_grid)
    lon_curves[lon] = (xs_grid, ys_grid)
    ax1.plot(
        xs_grid, ys_grid,
        color="gray", linewidth=0.5, linestyle="--", alpha=0.6,
        zorder=0, clip_on=True
    )

# Latitude curves
for lat in lat_ticks:
    lons_grid = np.linspace(50, 160, 1500)
    xs_grid, ys_grid = to_proj.transform(lons_grid, np.full_like(lons_grid, lat))
    lat_curves[lat] = (xs_grid, ys_grid)
    ax1.plot(
        xs_grid, ys_grid,
        color="gray", linewidth=0.5, linestyle="--", alpha=0.6,
        zorder=0, clip_on=True
    )

# ============================================================
# Find tick positions on the map frame
# ============================================================
lon_tick_x = []
for lon in lon_ticks:
    xs_grid, ys_grid = lon_curves[lon]
    idx = np.argmin(np.abs(ys_grid - main_ylim[0]))
    lon_tick_x.append(xs_grid[idx])

lat_tick_y = []
for lat in lat_ticks:
    xs_grid, ys_grid = lat_curves[lat]
    idx = np.argmin(np.abs(xs_grid - main_xlim[0]))
    lat_tick_y.append(ys_grid[idx])

# Longitude labels
ax1.set_xticks(lon_tick_x)
ax1.set_xticklabels([f"{lon}°E" for lon in lon_ticks], fontsize=FONT_SIZE)

# Latitude labels
ax1.set_yticks(lat_tick_y)
ax1.set_yticklabels([f"{lat}°N" for lat in lat_ticks], fontsize=FONT_SIZE)

ax1.tick_params(
    axis="both", which="major", direction="out",
    length=4, width=0.8, labelsize=FONT_SIZE,
    bottom=True, left=True, top=False, right=False
)

# ============================================================
# Panel A: border
# ============================================================
for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

# ============================================================
# Panel A: legend
# ============================================================
legend_handles = [
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor="blue", markeredgecolor="white", markersize=7,
        label=f"Data Center"
    )
]

ax1.legend(
    handles=legend_handles,
    loc="lower left",
    bbox_to_anchor=(-0.01, -0.028),
    fontsize=15,
    frameon=False
)

# ============================================================
# Panel A: curtailment colorbar (manual cax, doesn't touch ax1)
# ============================================================
cax = fig.add_axes([
    ax1_pos.x1 - 0.348,
    ax1_pos.y0 + ax1_pos.height * 0.081,
    0.01,
    ax1_pos.height * 0.3
])

sm = plt.cm.ScalarMappable(
    cmap="YlOrRd",
    norm=plt.Normalize(
        vmin=gdf_proj["curtailment_GWh"].min(),
        vmax=gdf_proj["curtailment_GWh"].max()
    )
)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.locator = MaxNLocator(nbins=5)



formatter = ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))  

cbar.formatter = formatter
cbar.update_ticks()

offset_text = cbar.ax.yaxis.get_offset_text()
offset_text.set_position((0, 1.04))
offset_text.set_ha("left")
offset_text.set_fontsize(15)

cbar.ax.tick_params(labelsize=15) 
cbar.set_label("")  
cax.text(
    0, -0.12,                          
    "2023-2025 Cumulative Curtailment (GWh)",
    transform=cax.transAxes,
    ha="left", va="bottom",
    fontsize=15
)
vmin = gdf_proj["curtailment_GWh"].min()
vmax = gdf_proj["curtailment_GWh"].max()


# ============================================================
# South China Sea inset
# ============================================================
inset_xlim_ll = [105, 122]
inset_ylim_ll = [2, 25]

ix0, iy0 = to_proj.transform(inset_xlim_ll[0], inset_ylim_ll[0])
ix1, iy1 = to_proj.transform(inset_xlim_ll[1], inset_ylim_ll[1])

inset_xlim = (min(ix0, ix1), max(ix0, ix1))
inset_ylim = (min(iy0, iy1), max(iy0, iy1))


inset_w = ax1_pos.width * 0.15
inset_h = ax1_pos.height * 0.28

inset_left = ax1_pos.x1 - inset_w - ax1_pos.width * 0.005
inset_bottom = ax1_pos.y0 + ax1_pos.height * 0.005

ax_inset = fig.add_axes([inset_left, inset_bottom, inset_w, inset_h])

# Inset map
province_proj.plot(
    ax=ax_inset, color="#F5F5F5", edgecolor="#333333", linewidth=0.3, zorder=4
)

ten_lines_buffer2.plot(
    ax=ax_inset, alpha=1, color="#c4cae3", edgecolor="none", zorder=1
)

ten_lines_buffer1.plot(
    ax=ax_inset, alpha=1, color="#aea8cd", edgecolor="none", zorder=2
)

ten_lines_proj.plot(ax=ax_inset, color="black", linewidth=1.0, zorder=5)
land_boundary_proj.plot(ax=ax_inset, color="black", linewidth=0.5, zorder=6)

ax_inset.set_xlim(inset_xlim)
ax_inset.set_ylim(inset_ylim)

# FIX 2 applied here too, for consistency
ax_inset.set_aspect("equal", adjustable="datalim")

ax_inset.set_xticks([])
ax_inset.set_yticks([])

for spine in ax_inset.spines.values():
    spine.set_edgecolor("#333333")
    spine.set_linewidth(1.0)

# ============================================================
# Panel B: horizontal grouped bar chart
# ============================================================
years = pivot.columns.tolist()

colors = ["#c6e5e8", "#6da4c3", "#385096"]

y = np.arange(len(pivot))
h = 0.8 / len(years)

for i, yr in enumerate(years):
    ax2.barh(
        y + i * h,
        pivot[yr].fillna(0) * 100,
        height=h,
        label=str(yr),
        color=colors[i % len(colors)]
    )

ax2.set_yticks(y + h * (len(years) - 1) / 2)
ax2.set_yticklabels(labels, fontsize=FONT_SIZE)
ax2.invert_yaxis()

ax2.set_xlabel("Curtailment Rate (%)", fontsize=FONT_SIZE)
ax2.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
ax2.legend(title="Year", fontsize=FONT_SIZE, title_fontsize=FONT_SIZE)

# ============================================================
# Panel B: border
# ============================================================
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

# ============================================================
# Final: force both panel positions back to exactly the same
# original positions
# ============================================================
# ============================================================
# Panel labels (A / B)
# ============================================================
ax1.text(
    -0.02, 1.02, "A",
    transform=ax1.transAxes,
    fontsize=18,
    fontweight="bold",
    ha="right", va="bottom"
)

ax2.text(
    -0.02, 1.02, "B",
    transform=ax2.transAxes,
    fontsize=18,
    fontweight="bold",
    ha="right", va="bottom"
)


fig.canvas.draw()

ax1.set_position(ax1_pos)
ax2.set_position(ax2_pos)

plt.savefig(str(RESULTS_ROOT / 'Figure/China_Map_Curtailment.png'), dpi=300, bbox_inches="tight")
plt.show()

# Figure 4: Provincial energy intensity and token potential

In [ ]:
import os
import re
import geopandas as gpd
import pandas as pd
import numpy as np
import numpy.ma as ma
import openpyxl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter

mpl.rcParams['font.sans-serif'] = ['Microsoft YaHei']
mpl.rcParams['axes.unicode_minus'] = False

gdf = gpd.read_file(str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/GS(2024)0650-SHP/sheng.shp'))

def normalize_code(x) -> str:
    'Normalize code.'
    return str(int(float(x)))

gdf['province_code'] = gdf['sheng_code'].apply(normalize_code)


EN_TO_CODE = {
    'beijing': '110000', 'tianjin': '120000', 'hebei': '130000', 'shanxi': '140000',
    'neimeng': '150000', 'liaoning': '210000', 'jilin': '220000', 'heilongjiang': '230000',
    'shanghai': '310000', 'jiangsu': '320000', 'zhejiang': '330000', 'anhui': '340000',
    'fujian': '350000', 'jiangxi': '360000', 'shandong': '370000', 'henan': '410000',
    'hubei': '420000', 'hunan': '430000', 'guangdong': '440000', 'guangxi': '450000',
    'hainan': '460000', 'chongqing': '500000', 'sichuan': '510000', 'guizhou': '520000',
    'yunnan': '530000', 'xizang': '540000', 'shannxi': '610000', 'gansu': '620000',
    'qinghai': '630000', 'ningxia': '640000', 'xinjiang': '650000',
}

EN_DISPLAY_NAME = {
    'beijing': 'Beijing', 'tianjin': 'Tianjin', 'hebei': 'Hebei', 'shanxi': 'Shanxi',
    'neimeng': 'Inner Mongolia', 'liaoning': 'Liaoning', 'jilin': 'Jilin', 'heilongjiang': 'Heilongjiang',
    'shanghai': 'Shanghai', 'jiangsu': 'Jiangsu', 'zhejiang': 'Zhejiang', 'anhui': 'Anhui',
    'fujian': 'Fujian', 'jiangxi': 'Jiangxi', 'shandong': 'Shandong', 'henan': 'Henan',
    'hubei': 'Hubei', 'hunan': 'Hunan', 'guangdong': 'Guangdong', 'guangxi': 'Guangxi',
    'hainan': 'Hainan', 'chongqing': 'Chongqing', 'sichuan': 'Sichuan', 'guizhou': 'Guizhou',
    'yunnan': 'Yunnan', 'xizang': 'Tibet', 'shannxi': 'Shaanxi', 'gansu': 'Gansu',
    'qinghai': 'Qinghai', 'ningxia': 'Ningxia', 'xinjiang': 'Xinjiang',
}
code_to_en = {v: k for k, v in EN_TO_CODE.items()}

In [ ]:
ENERGY_PATH = str(DATA_ROOT / '5_Data_Center/PUE/Summary/4_province_total_energy_per_million_tokens.csv')

energy_df = pd.read_csv(ENERGY_PATH, dtype={'province': str})
energy_df['province_code'] = energy_df['province'].apply(normalize_code)

map_df = gdf.merge(
    energy_df[['province_code', 'total_energy_S3_kWh_per_M_tokens']],
    on='province_code', how='left'
)

In [ ]:
CURTAILMENT_PATH = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/province_annual_curtailment.xlsx')
SHEET_NAME = 'Annual curtailment by province'
HEADER_ROW = 4
YEAR_COL = {2023: 3, 2024: 4, 2025: 5}
TARGET_YEAR = 2025
POTENTIAL_UTILIZATION_RATE = 0.70  

wb = openpyxl.load_workbook(CURTAILMENT_PATH, data_only=True)
ws = wb[SHEET_NAME]

def font_hex(cell):
    color = cell.font.color
    if color is None or not isinstance(color.rgb, str):
        return None
    return color.rgb[-6:].upper()

records = []
r = HEADER_ROW + 1
while True:
    prov_en = ws.cell(row=r, column=1).value
    if not isinstance(prov_en, str) or not re.fullmatch(r'[a-zA-Z_]+', prov_en.strip()):
        break

    prov_en = prov_en.strip().lower()
    cell = ws.cell(row=r, column=YEAR_COL[TARGET_YEAR])
    val = cell.value

    if val is None or (isinstance(val, str) and 'No data' in val):
        gwh, completeness = np.nan, 'none'
    else:
        gwh = float(val)
        completeness = 'partial' if font_hex(cell) == 'B45F06' else 'both'

    records.append({'province_en': prov_en, 'curtailment_2025_GWh': gwh, 'completeness_2025': completeness})
    r += 1

curtail_df = pd.DataFrame(records)
curtail_df['province_code'] = curtail_df['province_en'].map(EN_TO_CODE)

missing_code = curtail_df[curtail_df['province_code'].isna()]
if len(missing_code):
    print('Province:')
    print(missing_code[['province_en']].to_string(index=False))

curtail_2025 = curtail_df.dropna(subset=['province_code'])[
    ['province_code', 'curtailment_2025_GWh', 'completeness_2025']
]

map_df = map_df.merge(curtail_2025, on='province_code', how='left')
map_df['green_token_trillion_2025'] = (
    map_df['curtailment_2025_GWh'] * POTENTIAL_UTILIZATION_RATE / map_df['total_energy_S3_kWh_per_M_tokens']
)

In [ ]:
import numpy as np
import pyproj
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.close("all")

FONT_SIZE = 15
LAND_SIMPLIFY = 0.05
ZOOM = 1.12

CACHE_PATH = str(DATA_ROOT / 'external/BasicData/ChinaMap/cache.gpkg')
base_path = str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/GS(2024)0650-SHP')

china_proj_crs = (
    "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

# ============================================================

# ============================================================
province_proj = gpd.read_file(f"{base_path}/sheng.shp", engine="pyogrio").to_crs(china_proj_crs)
province_proj["geometry"] = province_proj.geometry.make_valid()

ten_lines_proj = gpd.read_file(f"{base_path}/tenlines.shp", engine="pyogrio").to_crs(china_proj_crs)

land_boundary_raw = gpd.read_file(f"{base_path}/continentline.shp")
land_boundary_raw["geometry"] = land_boundary_raw.geometry.simplify(tolerance=LAND_SIMPLIFY, preserve_topology=True)
land_boundary_proj = land_boundary_raw.to_crs(china_proj_crs)

ten_lines_buffer1 = gpd.read_file(CACHE_PATH, layer="ten_lines_buffer1", engine="pyogrio").to_crs(china_proj_crs)
ten_lines_buffer2 = gpd.read_file(CACHE_PATH, layer="ten_lines_buffer2", engine="pyogrio").to_crs(china_proj_crs)
land_boundary_buffer1 = gpd.read_file(CACHE_PATH, layer="land_boundary_buffer1", engine="pyogrio").to_crs(china_proj_crs)
land_boundary_buffer2 = gpd.read_file(CACHE_PATH, layer="land_boundary_buffer2", engine="pyogrio").to_crs(china_proj_crs)

map_df_proj = map_df.to_crs(china_proj_crs)

to_proj = pyproj.Transformer.from_crs("EPSG:4326", china_proj_crs, always_xy=True)

# ============================================================

# ============================================================
main_xlim_ll = [78.5, 132]
main_ylim_ll = [15, 55]
lon_ticks = [80, 90, 100, 110, 120, 130]
lat_ticks = [20, 30, 40, 50]

lons = np.linspace(main_xlim_ll[0], main_xlim_ll[1], 500)
lats = np.linspace(main_ylim_ll[0], main_ylim_ll[1], 500)
lon_boundary = np.concatenate([lons, lons, np.full_like(lats, main_xlim_ll[0]), np.full_like(lats, main_xlim_ll[1])])
lat_boundary = np.concatenate([np.full_like(lons, main_ylim_ll[0]), np.full_like(lons, main_ylim_ll[1]), lats, lats])
xs, ys = to_proj.transform(lon_boundary, lat_boundary)
xmin0, xmax0, ymin0, ymax0 = xs.min(), xs.max(), ys.min(), ys.max()
xcenter, ycenter = (xmin0 + xmax0) / 2, (ymin0 + ymax0) / 2

# ============================================================

# ============================================================
fig = plt.figure(figsize=(20, 16))

GRID = (100, 100)
ax1 = plt.subplot2grid(GRID, (0, 0),  rowspan=45, colspan=40)   
ax2 = plt.subplot2grid(GRID, (0, 45), rowspan=45, colspan=40)   
ax3 = plt.subplot2grid(GRID, (50, 0), rowspan=8, colspan=85)   

fig.canvas.draw()
ax1_pos = ax1.get_position().frozen()
ax2_pos = ax2.get_position().frozen()
ax3_pos = ax3.get_position().frozen()

fig_w, fig_h = fig.get_size_inches()

# ============================================================

# ============================================================
land_boundary_buffer2.plot(ax=ax1, alpha=1, color="#c4cae3", edgecolor="none", zorder=1)
land_boundary_buffer1.plot(ax=ax1, alpha=1, color="#aea8cd", edgecolor="none", zorder=2)
ten_lines_buffer2.plot(ax=ax1, alpha=1, color="#c4cae3", edgecolor="none", zorder=1)
ten_lines_buffer1.plot(ax=ax1, alpha=1, color="#aea8cd", edgecolor="none", zorder=2)

map_df_proj.plot(
    column='total_energy_S3_kWh_per_M_tokens', cmap='YlOrRd', linewidth=0.5, edgecolor="grey",
    ax=ax1, legend=False, missing_kwds={"color": "#e8e8e8"}, zorder=3
)

land_boundary_proj.plot(ax=ax1, linewidth=1.0, color="black", zorder=4)
ten_lines_proj.plot(ax=ax1, linewidth=1.0, color="black", zorder=5)

panel_ratio = (ax1_pos.width * fig_w) / (ax1_pos.height * fig_h)
target_height = (ymax0 - ymin0) * ZOOM
target_width = target_height * panel_ratio
xlim = (xcenter - target_width / 2, xcenter + target_width / 2)
ylim = (ycenter - target_height / 2, ycenter + target_height / 2)

ax1.set_xlim(xlim)
ax1.set_ylim(ylim)
ax1.set_aspect("equal", adjustable="datalim")
ax1.set_position(ax1_pos)


fig.canvas.draw()
xlim = ax1.get_xlim()
ylim = ax1.get_ylim()

lon_curves, lat_curves = {}, {}
for lon in lon_ticks:
    lats_grid = np.linspace(-10, 70, 1500)
    xg, yg = to_proj.transform(np.full_like(lats_grid, lon), lats_grid)
    lon_curves[lon] = (xg, yg)
    ax1.plot(xg, yg, color="gray", linewidth=0.5, linestyle="--", alpha=0.6, zorder=0, clip_on=True)
for lat in lat_ticks:
    lons_grid = np.linspace(50, 160, 1500)
    xg, yg = to_proj.transform(lons_grid, np.full_like(lons_grid, lat))
    lat_curves[lat] = (xg, yg)
    ax1.plot(xg, yg, color="gray", linewidth=0.5, linestyle="--", alpha=0.6, zorder=0, clip_on=True)

lon_tick_x = []
for lon in lon_ticks:
    xg, yg = lon_curves[lon]
    order = np.argsort(yg)
    lon_tick_x.append(np.interp(ylim[0], yg[order], xg[order]))

lat_tick_y = []
for lat in lat_ticks:
    xg, yg = lat_curves[lat]
    order = np.argsort(xg)
    lat_tick_y.append(np.interp(xlim[0], xg[order], yg[order]))

ax1.set_xticks(lon_tick_x)
ax1.set_xticklabels([f"{lon}°E" for lon in lon_ticks], fontsize=FONT_SIZE)
ax1.set_yticks(lat_tick_y)
ax1.set_yticklabels([f"{lat}°N" for lat in lat_ticks], fontsize=FONT_SIZE)

ax1.set_xlim(xlim)
ax1.set_ylim(ylim)

ax1.tick_params(axis="both", which="major", direction="out", length=4, width=0.8,
                 labelsize=FONT_SIZE, bottom=True, left=True, top=False, right=False)

for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

cax1 = fig.add_axes([
    ax1_pos.x0 + ax1_pos.width * 0.02,
    ax1_pos.y0 + ax1_pos.height * 0.09,
    ax1_pos.width * 0.03,
    ax1_pos.height * 0.3
])
vals1 = map_df_proj['total_energy_S3_kWh_per_M_tokens']
sm1 = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=vals1.min(), vmax=vals1.max()))
sm1.set_array([])
cbar1 = fig.colorbar(sm1, cax=cax1)
cbar1.locator = MaxNLocator(nbins=5)
cbar1.update_ticks()
cbar1.ax.tick_params(labelsize=FONT_SIZE)
cbar1.set_label("")
cax1.text(0, -0.3, "Energy Intensity\n(kWh/M tokens)", transform=cax1.transAxes,
           ha="left", va="bottom", fontsize=FONT_SIZE)

inset_xlim_ll = [105, 122]
inset_ylim_ll = [2, 25]
ix0, iy0 = to_proj.transform(inset_xlim_ll[0], inset_ylim_ll[0])
ix1, iy1 = to_proj.transform(inset_xlim_ll[1], inset_ylim_ll[1])
inset_xlim = (min(ix0, ix1), max(ix0, ix1))
inset_ylim = (min(iy0, iy1), max(iy0, iy1))

inset_w = ax1_pos.width * 0.15
inset_h = ax1_pos.height * 0.28
inset_left = ax1_pos.x1 - inset_w - ax1_pos.width * 0.005
inset_bottom = ax1_pos.y0 + ax1_pos.height * 0.005

ax1_inset = fig.add_axes([inset_left, inset_bottom, inset_w, inset_h])
province_proj.plot(ax=ax1_inset, color="#F5F5F5", edgecolor="#333333", linewidth=0.3, zorder=4)
ten_lines_buffer2.plot(ax=ax1_inset, alpha=1, color="#c4cae3", edgecolor="none", zorder=1)
ten_lines_buffer1.plot(ax=ax1_inset, alpha=1, color="#aea8cd", edgecolor="none", zorder=2)
ten_lines_proj.plot(ax=ax1_inset, color="black", linewidth=1.0, zorder=5)
land_boundary_proj.plot(ax=ax1_inset, color="black", linewidth=0.5, zorder=6)
ax1_inset.set_xlim(inset_xlim)
ax1_inset.set_ylim(inset_ylim)
ax1_inset.set_aspect("equal", adjustable="datalim")
ax1_inset.set_xticks([])
ax1_inset.set_yticks([])
for spine in ax1_inset.spines.values():
    spine.set_edgecolor("#333333")
    spine.set_linewidth(1.0)

ax1.text(-0.02, 1.02, "A", transform=ax1.transAxes,
          fontsize=18, fontweight="bold", ha="right", va="bottom")


# ============================================================

# ============================================================
land_boundary_buffer2.plot(ax=ax2, alpha=1, color="#c4cae3", edgecolor="none", zorder=1)
land_boundary_buffer1.plot(ax=ax2, alpha=1, color="#aea8cd", edgecolor="none", zorder=2)
ten_lines_buffer2.plot(ax=ax2, alpha=1, color="#c4cae3", edgecolor="none", zorder=1)
ten_lines_buffer1.plot(ax=ax2, alpha=1, color="#aea8cd", edgecolor="none", zorder=2)

map_df_proj.plot(
    column='green_token_trillion_2025', cmap='Greens', linewidth=0.5, edgecolor="grey",
    ax=ax2, legend=False, missing_kwds={"color": "#e8e8e8"}, zorder=3
)

land_boundary_proj.plot(ax=ax2, linewidth=1.0, color="black", zorder=4)
ten_lines_proj.plot(ax=ax2, linewidth=1.0, color="black", zorder=5)

panel_ratio2 = (ax2_pos.width * fig_w) / (ax2_pos.height * fig_h)
target_width2 = target_height * panel_ratio2  
xlim = (xcenter - target_width2 / 2, xcenter + target_width2 / 2)
ylim = (ycenter - target_height / 2, ycenter + target_height / 2)

ax2.set_xlim(xlim)
ax2.set_ylim(ylim)
ax2.set_aspect("equal", adjustable="datalim")
ax2.set_position(ax2_pos)


fig.canvas.draw()
xlim = ax2.get_xlim()
ylim = ax2.get_ylim()

lon_curves, lat_curves = {}, {}
for lon in lon_ticks:
    lats_grid = np.linspace(-10, 70, 1500)
    xg, yg = to_proj.transform(np.full_like(lats_grid, lon), lats_grid)
    lon_curves[lon] = (xg, yg)
    ax2.plot(xg, yg, color="gray", linewidth=0.5, linestyle="--", alpha=0.6, zorder=0, clip_on=True)
for lat in lat_ticks:
    lons_grid = np.linspace(50, 160, 1500)
    xg, yg = to_proj.transform(lons_grid, np.full_like(lons_grid, lat))
    lat_curves[lat] = (xg, yg)
    ax2.plot(xg, yg, color="gray", linewidth=0.5, linestyle="--", alpha=0.6, zorder=0, clip_on=True)

lon_tick_x = []
for lon in lon_ticks:
    xg, yg = lon_curves[lon]
    order = np.argsort(yg)
    lon_tick_x.append(np.interp(ylim[0], yg[order], xg[order]))

lat_tick_y = []
for lat in lat_ticks:
    xg, yg = lat_curves[lat]
    order = np.argsort(xg)
    lat_tick_y.append(np.interp(xlim[0], xg[order], yg[order]))

ax2.set_xticks(lon_tick_x)
ax2.set_xticklabels([f"{lon}°E" for lon in lon_ticks], fontsize=FONT_SIZE)
ax2.set_yticks(lat_tick_y)
ax2.set_yticklabels([f"{lat}°N" for lat in lat_ticks], fontsize=FONT_SIZE)

ax2.set_xlim(xlim)
ax2.set_ylim(ylim)

ax2.tick_params(axis="both", which="major", direction="out", length=4, width=0.8,
                 labelsize=FONT_SIZE, bottom=True, left=True, top=False, right=False)

for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

cax2 = fig.add_axes([
    ax2_pos.x0 + ax2_pos.width * 0.02,
    ax2_pos.y0 + ax2_pos.height * 0.09,
    ax2_pos.width * 0.03,
    ax2_pos.height * 0.3
])
vals2 = map_df_proj['green_token_trillion_2025']
sm2 = plt.cm.ScalarMappable(cmap="Greens", norm=plt.Normalize(vmin=vals2.min(), vmax=vals2.max()))
sm2.set_array([])
cbar2 = fig.colorbar(sm2, cax=cax2)
cbar2.locator = MaxNLocator(nbins=5)

formatter2 = ScalarFormatter(useMathText=True)
formatter2.set_scientific(True)
formatter2.set_powerlimits((0, 0))  
cbar2.formatter = formatter2
cbar2.update_ticks()


fig.canvas.draw()
offset_text = cbar2.ax.yaxis.get_offset_text()
exponent_str = offset_text.get_text()
offset_text.set_visible(False)
cax2.text(-0.3, 1.01, exponent_str, transform=cax2.transAxes,
          fontsize=FONT_SIZE, ha="left", va="bottom")

cbar2.ax.tick_params(labelsize=FONT_SIZE)
cbar2.set_label("")
cax2.text(0, -0.3, "CET Output\n(Trillion tokens)", transform=cax2.transAxes,
           ha="left", va="bottom", fontsize=FONT_SIZE)

inset_w = ax2_pos.width * 0.15
inset_h = ax2_pos.height * 0.28
inset_left = ax2_pos.x1 - inset_w - ax2_pos.width * 0.005
inset_bottom = ax2_pos.y0 + ax2_pos.height * 0.005

ax2_inset = fig.add_axes([inset_left, inset_bottom, inset_w, inset_h])
province_proj.plot(ax=ax2_inset, color="#F5F5F5", edgecolor="#333333", linewidth=0.3, zorder=4)
ten_lines_buffer2.plot(ax=ax2_inset, alpha=1, color="#c4cae3", edgecolor="none", zorder=1)
ten_lines_buffer1.plot(ax=ax2_inset, alpha=1, color="#aea8cd", edgecolor="none", zorder=2)
ten_lines_proj.plot(ax=ax2_inset, color="black", linewidth=1.0, zorder=5)
land_boundary_proj.plot(ax=ax2_inset, color="black", linewidth=0.5, zorder=6)
ax2_inset.set_xlim(inset_xlim)
ax2_inset.set_ylim(inset_ylim)
ax2_inset.set_aspect("equal", adjustable="datalim")
ax2_inset.set_xticks([])
ax2_inset.set_yticks([])
for spine in ax2_inset.spines.values():
    spine.set_edgecolor("#333333")
    spine.set_linewidth(1.0)

ax2.text(-0.02, 1.02, "B", transform=ax2.transAxes,
          fontsize=18, fontweight="bold", ha="right", va="bottom")

# ============================================================

# ============================================================
heat_df = map_df[['province_code', 'total_energy_S3_kWh_per_M_tokens', 'green_token_trillion_2025']].copy()
heat_df['province_en'] = heat_df['province_code'].map(code_to_en)
heat_df['display_name'] = heat_df['province_en'].map(EN_DISPLAY_NAME)
heat_df = heat_df.dropna(subset=['display_name'])
heat_df = heat_df.sort_values('green_token_trillion_2025', ascending=False, na_position='last')

def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

heat_df['energy_norm'] = normalize(heat_df['total_energy_S3_kWh_per_M_tokens'])
heat_df['green_norm'] = normalize(heat_df['green_token_trillion_2025'])

n_prov = len(heat_df)

energy_row = ma.masked_invalid(heat_df[['energy_norm']].values.T)   # shape (1, n_prov)
green_row = ma.masked_invalid(heat_df[['green_norm']].values.T)     # shape (1, n_prov)

cmap_energy = plt.cm.YlOrRd.copy()
cmap_energy.set_bad(color='#e8e8e8')
cmap_green = plt.cm.Greens.copy()
cmap_green.set_bad(color='#e8e8e8')


ax3.imshow(energy_row, aspect='auto', cmap=cmap_energy, vmin=0, vmax=1,
           extent=[-0.5, n_prov - 0.5, 1, 0])

ax3.imshow(green_row, aspect='auto', cmap=cmap_green, vmin=0, vmax=1,
           extent=[-0.5, n_prov - 0.5, 2, 1])

ax3.set_xlim(-0.5, n_prov - 0.5)
ax3.set_ylim(2, 0)

ax3.set_xticks(range(n_prov))
ax3.set_xticklabels(heat_df['display_name'], fontsize=FONT_SIZE - 3, rotation=60, ha='right')
ax3.set_yticks([0.5, 1.5])
ax3.set_yticklabels(['Energy\nIntensity', 'CET'], fontsize=FONT_SIZE - 3)
ax3.tick_params(axis="both", which="major", length=0)

for spine in ax3.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

TEXT_THRESHOLD = 0.70  
for i in range(n_prov):
    energy_val = heat_df.iloc[i]['total_energy_S3_kWh_per_M_tokens']
    green_val = heat_df.iloc[i]['green_token_trillion_2025']
    energy_norm_val = heat_df.iloc[i]['energy_norm']
    green_norm_val = heat_df.iloc[i]['green_norm']

    energy_text = f'{energy_val:.2f}' if pd.notna(energy_val) else 'NA'   
    green_text = f'{green_val:.0f}' if pd.notna(green_val) else 'NA'      

    energy_color = 'white' if pd.notna(energy_norm_val) and energy_norm_val > TEXT_THRESHOLD else 'black'
    green_color = 'white' if pd.notna(green_norm_val) and green_norm_val > TEXT_THRESHOLD else 'black'

    ax3.text(i, 0.5, energy_text, ha='center', va='center', fontsize=10, color=energy_color)
    ax3.text(i, 1.5, green_text, ha='center', va='center', fontsize=10, color=green_color)

ax3.text(-0.02, 1.18, "C", transform=ax3.transAxes,
          fontsize=18, fontweight="bold", ha="right", va="bottom")


OUT_FIG_PATH = str(RESULTS_ROOT / 'Figure/energy_intensity_and_green_token_2025.png')
plt.savefig(OUT_FIG_PATH, dpi=300, bbox_inches="tight")
print(f"[Done] Figure saved -> {OUT_FIG_PATH}")

plt.show()

# Figure 3: Installed capacity across three scenarios

In [ ]:
import numpy as np
import numpy.ma as ma
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter, NullFormatter

plt.close("all")

FONT_SIZE = 15
TOP_N = 10  

# ============================================================

# ============================================================
CAPACITY_PATH = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/capacity/province_annual_installed_capacity.xlsx')

cap_df = pd.read_excel(CAPACITY_PATH)
cap_2025 = cap_df[cap_df["year"] == 2025].copy()


EN_DISPLAY_NAME = {
    'beijing': 'Beijing', 'tianjin': 'Tianjin', 'hebei': 'Hebei', 'shanxi': 'Shanxi',
    'neimeng': 'Inner Mongolia', 'liaoning': 'Liaoning', 'jilin': 'Jilin', 'heilongjiang': 'Heilongjiang',
    'shanghai': 'Shanghai', 'jiangsu': 'Jiangsu', 'zhejiang': 'Zhejiang', 'anhui': 'Anhui',
    'fujian': 'Fujian', 'jiangxi': 'Jiangxi', 'shandong': 'Shandong', 'henan': 'Henan',
    'hubei': 'Hubei', 'hunan': 'Hunan', 'guangdong': 'Guangdong', 'guangxi': 'Guangxi',
    'hainan': 'Hainan', 'chongqing': 'Chongqing', 'sichuan': 'Sichuan', 'guizhou': 'Guizhou',
    'yunnan': 'Yunnan', 'xizang': 'Tibet', 'shannxi': 'Shaanxi', 'gansu': 'Gansu',
    'qinghai': 'Qinghai', 'ningxia': 'Ningxia', 'xinjiang': 'Xinjiang',
}
EN_TO_CODE = {
    'beijing': '110000', 'tianjin': '120000', 'hebei': '130000', 'shanxi': '140000',
    'neimeng': '150000', 'liaoning': '210000', 'jilin': '220000', 'heilongjiang': '230000',
    'shanghai': '310000', 'jiangsu': '320000', 'zhejiang': '330000', 'anhui': '340000',
    'fujian': '350000', 'jiangxi': '360000', 'shandong': '370000', 'henan': '410000',
    'hubei': '420000', 'hunan': '430000', 'guangdong': '440000', 'guangxi': '450000',
    'hainan': '460000', 'chongqing': '500000', 'sichuan': '510000', 'guizhou': '520000',
    'yunnan': '530000', 'xizang': '540000', 'shannxi': '610000', 'gansu': '620000',
    'qinghai': '630000', 'ningxia': '640000', 'xinjiang': '650000',
}
cap_2025["display_name"] = cap_2025["province"].map(EN_DISPLAY_NAME)
cap_2025["province_code"] = cap_2025["province"].map(EN_TO_CODE)

scenario_cols = [
    "scenario1_high_installed_capacity_MW",
    "scenario2_typical_installed_capacity_MW",
    "scenario3_flexible_installed_capacity_MW",
]
scenario_labels = ["S1", "S2", "S3"]
scenario_short = ["S1", "S2", "S3"]

# ============================================================

# ============================================================
PUE_PATH = str(DATA_ROOT / '5_Data_Center/PUE/Summary/4_province_total_energy_per_million_tokens.csv')

pue_df = pd.read_csv(PUE_PATH, dtype={"province": str})
pue_df = pue_df.rename(columns={"province": "province_code", "min_max_average_PUE": "PUE"})

cap_2025 = cap_2025.merge(pue_df[["province_code", "PUE"]], on="province_code", how="left")

# ============================================================

# ============================================================
national_totals_gw = cap_2025[scenario_cols].sum() / 1000

# ============================================================

# ============================================================
fig = plt.figure(figsize=(18, 5))

GRID = (100, 103)
ax3 = plt.subplot2grid(GRID, (0, 0),  rowspan=90, colspan=15)   
ax1 = plt.subplot2grid(GRID, (0, 18), rowspan=90, colspan=53)   
ax2 = plt.subplot2grid(GRID, (0, 75), rowspan=90, colspan=28)   

# ============================================================

# ============================================================
top_df = cap_2025.sort_values(scenario_cols[0], ascending=False).head(TOP_N).copy()


gw_cols = [f"{c}_GW" for c in scenario_cols]
for c, gw_c in zip(scenario_cols, gw_cols):
    top_df[gw_c] = top_df[c] / 1000

n_prov = len(top_df)
n_scen = len(scenario_cols)

matrix = ma.masked_invalid(top_df[gw_cols].values.T)  # shape (n_scen, n_prov)

cmap = plt.cm.YlOrRd.copy()
cmap.set_bad(color="#e8e8e8")

vmin = np.nanmin(top_df[gw_cols].values)
vmax = np.nanmax(top_df[gw_cols].values)
norm = LogNorm(vmin=vmin, vmax=vmax)

im = ax1.imshow(
    matrix, aspect="auto", cmap=cmap, norm=norm,
    extent=[-0.5, n_prov - 0.5, n_scen, 0]
)

ax1.set_xlim(-0.5, n_prov - 0.5)
ax1.set_ylim(n_scen, 0)

ax1.set_xticks(range(n_prov))
ax1.set_xticklabels(top_df["display_name"], fontsize=FONT_SIZE - 3, rotation=60, ha="right")
ax1.set_yticks([i + 0.5 for i in range(n_scen)])
ax1.set_yticklabels(scenario_labels, fontsize=FONT_SIZE - 3)
ax1.tick_params(axis="both", which="major", length=0)

for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

TEXT_THRESHOLD = 0.70
log_min, log_max = np.log10(vmin), np.log10(vmax)

for i in range(n_scen):
    for j in range(n_prov):
        val = matrix[i, j]
        if ma.is_masked(val):
            continue
        norm_val = (np.log10(val) - log_min) / (log_max - log_min)
        text_color = "white" if norm_val > TEXT_THRESHOLD else "black"
        ax1.text(j, i + 0.5, f"{val:,.1f}", ha="center", va="center",
                  fontsize=12, color=text_color)


ax1.text(-0.002, 1.03, "B", transform=ax1.transAxes,
          fontsize=18, fontweight="bold", ha="right", va="bottom")

# ============================================================

# ============================================================
scenario3 = cap_2025["scenario3_flexible_installed_capacity_MW"].values / 1000
scenario2 = cap_2025["scenario2_typical_installed_capacity_MW"].values / 1000
provinces = cap_2025["display_name"].values
pue_vals = cap_2025["PUE"].values

pue_cmap = plt.cm.viridis
pue_norm = plt.Normalize(vmin=np.nanmin(pue_vals), vmax=np.nanmax(pue_vals))

sc = ax2.scatter(scenario3, scenario2, s=50, c=pue_vals, cmap=pue_cmap, norm=pue_norm,
                  edgecolor="white", linewidth=0.5, zorder=3)

lims = [min(scenario3.min(), scenario2.min()) * 0.8, max(scenario3.max(), scenario2.max()) * 1.2]
ax2.plot(lims, lims, linestyle="--", color="gray", linewidth=1.0, zorder=1, label="equal capacity")
ax2.legend(fontsize=FONT_SIZE - 2, frameon=False, loc="upper left")

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_aspect("equal")
ax2.set_xlabel("S3 Installed Capacity (GW)", fontsize=FONT_SIZE - 2)
ax2.set_ylabel("S2 Installed Capacity (GW)", fontsize=FONT_SIZE - 2, labelpad=-2)

ax2.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 3)
ax2.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:g}"))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:g}"))
ax2.xaxis.set_minor_formatter(NullFormatter())
ax2.yaxis.set_minor_formatter(NullFormatter())



ax2_pos = ax2.get_position()
cax_pue = fig.add_axes([
    ax2_pos.x1 - 0.045,
    ax2_pos.y0 + ax2_pos.height * 0.06,
    0.012,
    ax2_pos.height * 0.28
])

cbar_pue = fig.colorbar(sc, cax=cax_pue)


pue_min = np.nanmin(pue_vals)
pue_max = np.nanmax(pue_vals)
pue_mid = (pue_min + pue_max) / 2

cbar_pue.set_ticks([pue_min, pue_mid, pue_max])
cbar_pue.ax.set_yticklabels([f"{pue_min:.2f}", f"{pue_mid:.2f}", f"{pue_max:.2f}"])
cbar_pue.ax.tick_params(labelsize=FONT_SIZE - 4)


cbar_pue.ax.set_xlabel("PUE", fontsize=FONT_SIZE - 4, labelpad=4)

ax2.text(-0.02, 1.03, "C", transform=ax2.transAxes,
          fontsize=18, fontweight="bold", ha="right", va="bottom")

# ============================================================

# ============================================================
colors = ["#c6e5e8", "#6da4c3", "#385096"]

bars = ax3.bar(scenario_short, national_totals_gw.values, color=colors, width=0.6)

for bar, val in zip(bars, national_totals_gw.values):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              f"{val:,.0f}", ha="center", va="bottom", fontsize=FONT_SIZE - 3)

ax3.set_ylabel("National Total Capacity (GW)", fontsize=FONT_SIZE - 3)
ax3.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 3)
ax3.set_ylim(0, national_totals_gw.max() * 1.15)

for spine in ax3.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_edgecolor("black")

ax3.text(-0.05, 1.03, "A", transform=ax3.transAxes,
          fontsize=18, fontweight="bold", ha="right", va="bottom")

# ============================================================

# ============================================================
OUT_FIG_PATH = str(RESULTS_ROOT / 'Figure/installed_capacity_2025_summary.png')
plt.savefig(OUT_FIG_PATH, dpi=600, bbox_inches="tight")
print(f"[Done] Figure saved -> {OUT_FIG_PATH}")

plt.show()

# Figure 2: Capacity and renewable share across utilization targets

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
 
# ============ Paths and settings ============
SUMMARY_FILE = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/gradient/gradient_scenarios_summary.xlsx')
DAILY_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/daily')
FIG_FILE = str(RESULTS_ROOT / 'Figure/Gradient_scenarios.png')
YEAR = 2025
 
color_util, color_share = '#1f77b4', '#d62728'
DAILY_COLORS = ['#6DA8D6', '#A8D08D']
LW, MS = 1.5, 4
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})
 


ROWS = [
    ('scenario1_high',     'scenario1_70',    'S1', (105, 175), [0, 20, 40, 60, 80, 100, 180, 200],           dict(bbox_to_anchor=(0.5, 0.48))),
    ('scenario2_typical',  'scenario2_70',    'S2', (155, 315), [0, 30, 60, 90, 120, 150, 320, 350],              dict(bbox_to_anchor=(0.5, 0.40))),
    ('scenario3_flexible', 'scenario3new_70', 'S3', (85, 145),  [0, 20, 40, 60, 80, 150, 170],            dict(loc='best')),
]
GAP_FRAC = 0.05   
 
nat = pd.read_excel(SUMMARY_FILE, sheet_name='national')
nat = nat[nat['year'] == YEAR]
 
 

def make_gap_scale(a, b, xmax, frac=GAP_FRAC):
    'Make gap scale.'
    span = b - a
    k = frac * (xmax - span) / (span * (1 - frac))
 
    def fwd(x):
        x = np.asarray(x, dtype=float)
        return np.where(x < a, x, np.where(x <= b, a + (x - a) * k, x - span * (1 - k)))
 
    def inv(y):
        y = np.asarray(y, dtype=float)
        yb = a + span * k
        return np.where(y < a, y, np.where(y <= yb, a + (y - a) / k, y + span * (1 - k)))
 
    return fwd, inv
 
 
def draw_break(ax, a, b, fwd, xlim):
    'Draw break.'
    x0, x1 = fwd(xlim[0]), fwd(xlim[1])
    xa = float((fwd(a) - x0) / (x1 - x0))
    xb = float((fwd(b) - x0) / (x1 - x0))
    t = np.linspace(-1, 1, 80)
    amp = 0.006
    for yc in (0,):
        yy = yc + 0.045 * t
        left = xa + amp * np.sin(2 * np.pi * t)
        right = xb + amp * np.sin(2 * np.pi * t)
        ax.fill_betweenx(yy, left, right, transform=ax.transAxes, color='white',
                         clip_on=False, zorder=10, linewidth=0)
        for xx in (left, right):
            ax.plot(xx, yy, transform=ax.transAxes, color='black', linewidth=0.8,
                    clip_on=False, zorder=11)
 
 

def plot_gradient(ax, d, gap, ticks, legend_kw, show_xlabel):
    x = d['installed_capacity_MW'] / 1000
    a, b = gap
    xmax = x.max() * 1.03
    fwd, inv = make_gap_scale(a, b, xmax)
    ax.set_xscale('function', functions=(fwd, inv))
    ax.set_xlim(0, xmax)
 
    ax.plot(x, d['curtailment_utilization_%'], color=color_util, marker='o', linewidth=LW, markersize=MS, label='Curtailment utilization')
    ax.set_ylabel('Curtailment utilization (%)', color='black')
    ax.tick_params(axis='y')
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'{t:g}' for t in ticks])
    if show_xlabel:
        ax.set_xlabel('Installed capacity (GW)')
 
    axr = ax.twinx()
    axr.plot(x, d['dc_renewable_share_%'], color=color_share, marker='s', linewidth=LW, markersize=MS, label='Curtailed renewable share')
    axr.set_ylabel('Curtailment share (%)', color='black')
    axr.tick_params(axis='y')
    axr.set_ylim(0, 60)
    axr.set_yticks(np.arange(0, 61, 20))

    
    draw_break(axr, a, b, fwd, ax.get_xlim())
    return axr
 
 

def plot_daily(ax, scenario_key, show_xlabel):
    fname = f'daily_national_total_{YEAR}_{scenario_key}.xlsx'
    fpath = os.path.join(DAILY_DIR, fname)
    if not os.path.exists(fpath):
        ax.text(0.5, 0.5, f'Not found:\n{fname}', ha='center', va='center',
                transform=ax.transAxes, color='red', fontsize=9)
        ax.axis('off')
        return
 
    df = pd.read_excel(fpath)
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    x = df['date'].values
    y_baseload = df['baseload_MWh'].values / 1000
    y_greenload = df['greenload_MWh'].values / 1000
 
    ax.stackplot(x, y_baseload, y_greenload, colors=DAILY_COLORS, alpha=0.85)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(x[0], x[-1])
    ax.set_ylabel('Daily energy mix (GWh)')
    if show_xlabel:
        ax.set_xlabel('Month')
 
 
# ============ Figure and grid ============
fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(3, 2, width_ratios=[1, 1], hspace=0.25, wspace=0.30)
 
left_axes, right_axes = [], []
for i, (grad_key, daily_key, scen, gap, ticks, legend_kw) in enumerate(ROWS):
    is_last = i == len(ROWS) - 1
    d = nat[nat['scenario'] == grad_key].sort_values('installed_capacity_MW')
 
    axL = fig.add_subplot(gs[i, 0])
    plot_gradient(axL, d, gap, ticks, legend_kw, show_xlabel=is_last)
    left_axes.append(axL)
 
    axR = fig.add_subplot(gs[i, 1])
    plot_daily(axR, daily_key, show_xlabel=is_last)
    right_axes.append(axR)
 

legend_elements = [
    Patch(facecolor=DAILY_COLORS[0], alpha=0.85, label='Grid Power'),
    Patch(facecolor=DAILY_COLORS[1], alpha=0.85, label='Curtailed Power'),
]
right_axes[-1].legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.28),
                      ncol=2, frameon=False)


grad_legend = [
    Line2D([0], [0], color=color_util, marker='o', linewidth=LW, markersize=MS, label='Curtailment utilization'),
    Line2D([0], [0], color=color_share, marker='s', linewidth=LW, markersize=MS, label='Curtailed renewable share'),
]
left_axes[-1].legend(handles=grad_legend, loc='upper center', bbox_to_anchor=(0.5, -0.28),
                     ncol=2, frameon=False)

# ============ Panel labels (A–F) and scenario tags (S1/S2/S3) ============
for row, (axL, axR) in enumerate(zip(left_axes, right_axes)):
    scen = ROWS[row][2]
    for ax, panel in ((axL, 'ACE'[row]), (axR, 'BDF'[row])):
        ax.text(-0.08, 1.04, panel, transform=ax.transAxes,
                fontsize=12, fontweight='bold', va='bottom', ha='left')
        
        ax.text(-0.03, 1.04, scen, transform=ax.transAxes,
                fontsize=10, fontweight='bold', va='bottom', ha='left')
 
plt.savefig(FIG_FILE, dpi=300, bbox_inches='tight')
plt.show()

# Figure 5: Sensitivity of national token potential

# Cooling technology sensitivity

In [ ]:
# base token output

import re
import numpy as np
import pandas as pd
import openpyxl

# ---------- Paths and parameters (same as 6_Figure Block 2) ----------
ENERGY_PATH = str(DATA_ROOT / '5_Data_Center/PUE/Summary/4_province_total_energy_per_million_tokens.csv')
CURTAILMENT_PATH = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/province_annual_curtailment.xlsx')
SHEET_NAME = 'Annual curtailment by province'
HEADER_ROW = 4
YEAR_COL = {2023: 3, 2024: 4, 2025: 5}
TARGET_YEAR = 2025
POTENTIAL_UTILIZATION_RATE = 0.70

EN_TO_CODE = {
    'beijing': '110000', 'tianjin': '120000', 'hebei': '130000', 'shanxi': '140000',
    'neimeng': '150000', 'liaoning': '210000', 'jilin': '220000', 'heilongjiang': '230000',
    'shanghai': '310000', 'jiangsu': '320000', 'zhejiang': '330000', 'anhui': '340000',
    'fujian': '350000', 'jiangxi': '360000', 'shandong': '370000', 'henan': '410000',
    'hubei': '420000', 'hunan': '430000', 'guangdong': '440000', 'guangxi': '450000',
    'hainan': '460000', 'chongqing': '500000', 'sichuan': '510000', 'guizhou': '520000',
    'yunnan': '530000', 'xizang': '540000', 'shannxi': '610000', 'gansu': '620000',
    'qinghai': '630000', 'ningxia': '640000', 'xinjiang': '650000',
}

# ---------- Energy intensity per province (S3, kWh/M tokens) ----------
energy_df = pd.read_csv(ENERGY_PATH, dtype={'province': str})
energy_df['province_code'] = energy_df['province'].astype(float).astype(int).astype(str)

# ---------- Annual curtailment per province (GWh) ----------
wb = openpyxl.load_workbook(CURTAILMENT_PATH, data_only=True)
ws = wb[SHEET_NAME]

records = []
r = HEADER_ROW + 1
while True:
    prov_en = ws.cell(row=r, column=1).value
    if not isinstance(prov_en, str) or not re.fullmatch(r'[a-zA-Z_]+', prov_en.strip()):
        break

    prov_en = prov_en.strip().lower()
    cell = ws.cell(row=r, column=YEAR_COL[TARGET_YEAR])
    val = cell.value

    if val is None or (isinstance(val, str) and 'No data' in val):
        gwh, completeness = np.nan, 'none'
    else:
        gwh = float(val)
        font_color = cell.font.color
        font_hex = font_color.rgb[-6:].upper() if font_color is not None and isinstance(font_color.rgb, str) else None
        completeness = 'partial' if font_hex == 'B45F06' else 'both'   # orange font = solar only or wind only

    records.append({'province_en': prov_en, 'curtailment_GWh': gwh, 'completeness': completeness})
    r += 1

curtail_df = pd.DataFrame(records)
curtail_df['province_code'] = curtail_df['province_en'].map(EN_TO_CODE)

# ---------- Green token output (trillion tokens): base + min / max PUE ----------
# base      : IT load S3 x min_max_average_PUE
# PUEmin    : IT load S3 x min_PUE  -> upper bound of token output
# PUEmax    : IT load S3 x max_PUE  -> lower bound of token output
token_df = curtail_df.merge(
    energy_df[['province_code', 'min_PUE', 'min_max_average_PUE', 'max_PUE',
               'total_energy_S3_low_kWh_per_M_tokens',
               'total_energy_S3_kWh_per_M_tokens',
               'total_energy_S3_high_kWh_per_M_tokens']],
    on='province_code', how='left'
)

token_df['green_token_trillion'] = (
    token_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE / token_df['total_energy_S3_kWh_per_M_tokens']
)
token_df['green_token_trillion_PUEmin'] = (
    token_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE / token_df['total_energy_S3_low_kWh_per_M_tokens']
)
token_df['green_token_trillion_PUEmax'] = (
    token_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE / token_df['total_energy_S3_high_kWh_per_M_tokens']
)

token_df = token_df.sort_values('green_token_trillion', ascending=False, na_position='last').reset_index(drop=True)

out_cols = ['province_en', 'province_code', 'curtailment_GWh', 'completeness',
            'min_PUE', 'min_max_average_PUE', 'max_PUE',
            'green_token_trillion_PUEmax', 'green_token_trillion', 'green_token_trillion_PUEmin']
print(token_df[out_cols].to_string(index=False))

OUT_PATH = str(DATA_ROOT / '5_Data_Center/PUE/Summary/5_province_green_token_PUE_min_max_2025.csv')
token_df[out_cols].to_csv(OUT_PATH, index=False, float_format='%.6f')

# ---------- National totals ----------
tok_base = token_df['green_token_trillion'].sum()
tok_lo = token_df['green_token_trillion_PUEmax'].sum()
tok_hi = token_df['green_token_trillion_PUEmin'].sum()
n_all = token_df['green_token_trillion'].notna().sum()

print(f"\nTotal curtailment {TARGET_YEAR}: {token_df['curtailment_GWh'].sum():,.2f} GWh")
print(f"Total green token output {TARGET_YEAR} ({n_all} provinces, trillion tokens)")
print(f"  max PUE (lower): {tok_lo:,.2f}  ({(tok_lo / tok_base - 1) * 100:+.1f}% vs base)")
print(f"  base           : {tok_base:,.2f}  = {tok_base * 1e12:.3e} tokens")
print(f"  min PUE (upper): {tok_hi:,.2f}  ({(tok_hi / tok_base - 1) * 100:+.1f}% vs base)")
print(f"\n[Done] -> {OUT_PATH}")

tok_cooling_min = token_df['green_token_trillion_PUEmax'].sum()
tok_cooling_max = token_df['green_token_trillion_PUEmin'].sum()

# Thermodynamic model sensitivity

In [ ]:

# ---------- parameter sensivtivity ----------
LONG_PATH = str(DATA_ROOT / '5_Data_Center/PUE/Summary/1_all_cases_annual_mean_long.csv')
OUT_PATH = str(DATA_ROOT / '5_Data_Center/PUE/Summary/6_national_green_token_by_sample_2025.csv')
SELECTED_CASES = [1, 2, 3, 4, 5, 6, 7, 11, 12]
IT_LOAD_S3 = 0.840728 * 1.10 * 1.10 * 1.08

# Per province x sample: min / max across the 9 cases, then midpoint
case_long = pd.read_csv(LONG_PATH)
case_long = case_long[case_long['case'].isin(SELECTED_CASES)]
case_long['province_code'] = case_long['province'].astype(float).astype(int).astype(str)

pue_long = case_long.groupby(['province_code', 'sample'])['annual_mean'].agg(
    min_PUE='min', max_PUE='max', n_cases='count').reset_index()
pue_long['PUE'] = (pue_long['min_PUE'] + pue_long['max_PUE']) / 2

if (pue_long['n_cases'] != len(SELECTED_CASES)).any():
    print("[Warning] some province x sample combinations are missing cases")

# Token output per province x sample
sample_df = pue_long.merge(curtail_df[['province_code', 'curtailment_GWh']], on='province_code', how='inner')
sample_df = sample_df.dropna(subset=['curtailment_GWh', 'PUE'])
sample_df['total_energy_S3_kWh_per_M_tokens'] = IT_LOAD_S3 * sample_df['PUE']
sample_df['green_token_trillion'] = (
    sample_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE / sample_df['total_energy_S3_kWh_per_M_tokens']
)

# National total for each sample
national = sample_df.groupby('sample').agg(
    green_token_trillion=('green_token_trillion', 'sum'),
    n_provinces=('province_code', 'nunique'),
).reset_index()
print(national.to_string(index=False))

if national['n_provinces'].nunique() > 1:
    print("[Warning] number of provinces differs across samples")

tok = national['green_token_trillion']
print(f"\nNational green token output {TARGET_YEAR} across 50 samples (trillion tokens)")
print(f"  min    : {tok.min():,.2f}")
print(f"  p5     : {tok.quantile(0.05):,.2f}")
print(f"  median : {tok.median():,.2f}")
print(f"  p95    : {tok.quantile(0.95):,.2f}")
print(f"  max    : {tok.max():,.2f}")
print(f"  base   : {token_df['green_token_trillion'].sum():,.2f}")

tok_thermal_min = tok.min()
tok_thermal_max = tok.max()

#national.to_csv(OUT_PATH, index=False, float_format='%.6f')
#print(f"\n[Done] -> {OUT_PATH}")

# Line-of-sight efficiency improvements

In [ ]:
# # S3 Line-of-sight sensitivity: energy per million tokens with improvement pathways
 
#%%
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.linear_model import LinearRegression
 
#%%
# ---------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------
seed = 42
n_runs = 100000
TOKENS_PER_UNIT = 1_000_000          # per million output tokens
NODE_POWER_KW = 10.2
PU_RANGE = (0.4, 0.9)                # power utilization, (P5, P95) for lognormal
 
INPUT_LENGTH_MEDIAN = 900
OUTPUT_MEDIANS = {'traditional': 300, 'test_time_scaling': 5000}
WEIGHT_TRADITIONAL = 0.90
WEIGHT_TTS = 0.10
 
OVERHEAD_S3 = 1.10 * 1.10 * 1.08     # network fabric, node redundancy, idle nodes (same as S2)
 
# Improvement multipliers, (P5, P95) for lognormal
IMPROVEMENT_RANGES = {
    'model':    (1.5, 10),    # Algorithm / model
    'serving':  (1.5, 5),     # Serving platform
    'hardware': (1.5, 2.5),   # Hardware & datacenter
}
 
TPS_PATH = str(DATA_ROOT / '5_Data_Center/TPS_node/TPS_20260821.xlsx')
OUT_DIR = str(DATA_ROOT / '5_Data_Center/PUE/Summary')
PUE_PATH = os.path.join(OUT_DIR, '3_province_min_max_average_PUE.csv')
SHP_PATH = str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/2019\u4e2d\u56fd\u5730\u56fe-\u5ba1\u56fe\u53f7GS(2019)1822\u53f7/\u7701_WGS84.shp')
OUT_PATH = os.path.join(OUT_DIR, '4_province_total_energy_per_million_tokens_S3_sensitivity.csv')
 
# Lognormal (mu, sigma) from (P5, P95): sigma = (ln P95 - ln P5) / (2 * 1.645)
sigma_pu = (np.log(PU_RANGE[1]) - np.log(PU_RANGE[0])) / (2 * 1.645)
mu_pu = np.log(PU_RANGE[0]) + 1.645 * sigma_pu
 
input_scale = INPUT_LENGTH_MEDIAN / np.log(2)                       # exponential scale from median
output_scales = {k: m / np.log(2) for k, m in OUTPUT_MEDIANS.items()}
 
#%%
# ---------------------------------------------------------------------
# TPS regression per model: log(TPS) ~ log(input) + log(output)
# (models with 2 points fall back to max observed TPS)
# ---------------------------------------------------------------------
tps_df = pd.read_excel(TPS_PATH)
tps_df['TPS_numeric'] = pd.to_numeric(tps_df['TPS'], errors='coerce')
tps_df['Input_Length_numeric'] = pd.to_numeric(tps_df['Input Length'], errors='coerce')
tps_df['Output_Length_numeric'] = pd.to_numeric(tps_df['Output Length'], errors='coerce')
 
tps_models = {}
for model_name in tps_df['Model'].unique():
    subset = tps_df[tps_df['Model'] == model_name].dropna(
        subset=['Input_Length_numeric', 'Output_Length_numeric', 'TPS_numeric'])
    if len(subset) < 2:
        continue
    max_tps = subset['TPS_numeric'].max()
    if len(subset) < 3:
        tps_models[model_name] = {'type': 'max_tps', 'max_tps': max_tps}
    else:
        X_log = np.log(np.maximum(subset[['Input_Length_numeric', 'Output_Length_numeric']].values, 1e-6))
        y_log = np.log(np.maximum(subset['TPS_numeric'].values, 1e-6))
        reg = LinearRegression().fit(X_log, y_log)
        tps_models[model_name] = {'type': 'regression', 'model': reg, 'max_tps': max_tps}
        print(f"{model_name:35s} n={len(subset):2d}  R²(log)={reg.score(X_log, y_log):.4f}")
 
#%%
# ---------------------------------------------------------------------
# Baseline Monte Carlo: per-query IT-load energy (Wh), no PUE
# (same sampling order and seed as 4_EnergyperToken.ipynb)
# ---------------------------------------------------------------------
baseline_energy_wh = {}     # (scenario, model) -> array
baseline_out_tokens = {}    # (scenario, model) -> array
 
for scenario_key in OUTPUT_MEDIANS:
    np.random.seed(seed)
    for model_name, info in tps_models.items():
        input_lengths = np.maximum(np.round(np.random.exponential(input_scale, n_runs)).astype(int), 1)
        output_lengths = np.maximum(np.round(np.random.exponential(output_scales[scenario_key], n_runs)).astype(int), 1)
 
        if info['type'] == 'regression':
            log_features = np.column_stack([np.log(input_lengths), np.log(output_lengths)])
            tps = np.minimum(np.exp(info['model'].predict(log_features)), info['max_tps'])
        else:
            tps = np.full(n_runs, info['max_tps'])
 
        pu = np.random.lognormal(mu_pu, sigma_pu, n_runs)
 
        energy_wh = (NODE_POWER_KW * pu * output_lengths / tps) / 3600 * 1000
        baseline_energy_wh[(scenario_key, model_name)] = energy_wh
        baseline_out_tokens[(scenario_key, model_name)] = output_lengths
 
# Baseline weighted IT load (should reproduce ~0.840728 kWh/M tokens)
baseline_rows = []
for (scenario_key, model_name), energy_wh in baseline_energy_wh.items():
    tokens = baseline_out_tokens[(scenario_key, model_name)]
    baseline_rows.append({
        'scenario': scenario_key,
        'model': model_name,
        'kWh_per_M_tokens': energy_wh.sum() / tokens.sum() * TOKENS_PER_UNIT / 1000,
    })
baseline_df = pd.DataFrame(baseline_rows)
baseline_trad = baseline_df.loc[baseline_df['scenario'] == 'traditional', 'kWh_per_M_tokens'].mean()
baseline_tts = baseline_df.loc[baseline_df['scenario'] == 'test_time_scaling', 'kWh_per_M_tokens'].mean()
baseline_weighted = baseline_trad * WEIGHT_TRADITIONAL + baseline_tts * WEIGHT_TTS
 
print(baseline_df.to_string(index=False))
print(f"\nBaseline weighted IT load (90/10): {baseline_weighted:.6f} kWh/M tokens")
 
#%%
# ---------------------------------------------------------------------
# S3: divide each simulated query's energy by the pathway multiplier,
# then ratio-of-sums -> mean across models -> 90/10 weighting -> x overhead
# ---------------------------------------------------------------------
rng_imp = np.random.default_rng(seed)
 
s3_rows = []
for pathway, (p5_imp, p95_imp) in IMPROVEMENT_RANGES.items():
    sigma_imp = (np.log(p95_imp) - np.log(p5_imp)) / (2 * 1.645)
    mu_imp = np.log(p5_imp) + 1.645 * sigma_imp
    for (scenario_key, model_name), energy_wh in baseline_energy_wh.items():
        tokens = baseline_out_tokens[(scenario_key, model_name)]
        multiplier = rng_imp.lognormal(mu_imp, sigma_imp, len(energy_wh))
        energy_wh_improved = energy_wh / multiplier
        s3_rows.append({
            'pathway': pathway,
            'scenario': scenario_key,
            'model': model_name,
            'multiplier_median': np.median(multiplier),
            'kWh_per_M_tokens': energy_wh_improved.sum() / tokens.sum() * TOKENS_PER_UNIT / 1000,
        })
 
s3_df = pd.DataFrame(s3_rows)
print(s3_df.to_string(index=False))
 
IT_LOAD_KWH_PER_M_TOKENS_S3 = {}
print()
for pathway in IMPROVEMENT_RANGES:
    sub = s3_df[s3_df['pathway'] == pathway]
    trad_avg = sub.loc[sub['scenario'] == 'traditional', 'kWh_per_M_tokens'].mean()
    tts_avg = sub.loc[sub['scenario'] == 'test_time_scaling', 'kWh_per_M_tokens'].mean()
    weighted = trad_avg * WEIGHT_TRADITIONAL + tts_avg * WEIGHT_TTS
    IT_LOAD_KWH_PER_M_TOKENS_S3[pathway] = weighted * OVERHEAD_S3
    print(f"S3_{pathway:9s} weighted IT load: {weighted:.6f} kWh/M tokens | "
          f"x overhead: {IT_LOAD_KWH_PER_M_TOKENS_S3[pathway]:.6f} | "
          f"reduction vs baseline: {(1 - weighted / baseline_weighted) * 100:.1f}%")
 
print(f"\nReference S2 (no improvement): {baseline_weighted * OVERHEAD_S3:.6f} kWh/M tokens")

In [ ]:
# ---------------------------------------------------------------------
# Line-of-sight: baseline vs. three improvement pathways (green token output)
# Reuses token_df from the base cell; only the IT load changes
# ---------------------------------------------------------------------
LOS_PROVINCE_OUT = str(DATA_ROOT / '5_Data_Center/PUE/Summary/7_province_green_token_line_of_sight_2025.csv')

token_df['green_token_trillion_model'] = (
    token_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE
    / (IT_LOAD_KWH_PER_M_TOKENS_S3['model'] * token_df['min_max_average_PUE'])
)
token_df['green_token_trillion_serving'] = (
    token_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE
    / (IT_LOAD_KWH_PER_M_TOKENS_S3['serving'] * token_df['min_max_average_PUE'])
)
token_df['green_token_trillion_hardware'] = (
    token_df['curtailment_GWh'] * POTENTIAL_UTILIZATION_RATE
    / (IT_LOAD_KWH_PER_M_TOKENS_S3['hardware'] * token_df['min_max_average_PUE'])
)

los_cols = ['province_en', 'province_code', 'curtailment_GWh', 'green_token_trillion',
            'green_token_trillion_model', 'green_token_trillion_serving', 'green_token_trillion_hardware']
print(token_df[los_cols].to_string(index=False))

tok_base = token_df['green_token_trillion'].sum()
tok_model = token_df['green_token_trillion_model'].sum()
tok_serving = token_df['green_token_trillion_serving'].sum()
tok_hardware = token_df['green_token_trillion_hardware'].sum()

print(f"\nNational green token output {TARGET_YEAR} (trillion tokens)")
print(f"  baseline  {tok_base:10,.2f}")
print(f"  model     {tok_model:10,.2f}  x{tok_model / tok_base:.2f} vs baseline")
print(f"  serving   {tok_serving:10,.2f}  x{tok_serving / tok_base:.2f} vs baseline")
print(f"  hardware  {tok_hardware:10,.2f}  x{tok_hardware / tok_base:.2f} vs baseline")

token_df[los_cols].to_csv(LOS_PROVINCE_OUT, index=False, float_format='%.6f')
print(f"\n[Done] -> {LOS_PROVINCE_OUT}")

In [ ]:
from matplotlib.patches import Patch

# ---------- % change relative to baseline ----------
pct_cooling_min = (tok_cooling_min / tok_base - 1) * 100
pct_cooling_max = (tok_cooling_max / tok_base - 1) * 100
pct_thermal_min = (tok_thermal_min / tok_base - 1) * 100
pct_thermal_max = (tok_thermal_max / tok_base - 1) * 100
pct_model = (tok_model / tok_base - 1) * 100
pct_serving = (tok_serving / tok_base - 1) * 100
pct_hardware = (tok_hardware / tok_base - 1) * 100

print(f"Cooling type        : {pct_cooling_min:+.1f}%  {pct_cooling_max:+.1f}%")
print(f"Thermodynamic model : {pct_thermal_min:+.1f}%  {pct_thermal_max:+.1f}%")
print(f"LOS model           : {pct_model:+.1f}%")
print(f"LOS serving         : {pct_serving:+.1f}%")
print(f"LOS hardware        : {pct_hardware:+.1f}%")

# ---------- Figure ----------
fig = plt.figure(figsize=(11, 6))
ax = plt.subplot(1, 1, 1)
plt.subplots_adjust(left=0.04, right=0.96, top=0.70, bottom=0.13)

y = [4, 3, 2, 1, 0]
colors = ['#2E75A0', '#EBB320', '#82B85A', '#4BAFAF', '#A04A7A']
low = [pct_cooling_min, pct_thermal_min, 0, 0, 0]
high = [pct_cooling_max, pct_thermal_max, pct_model, pct_serving, pct_hardware]

ax.barh(y, low, height=0.75, color=colors, edgecolor='none')
ax.barh(y, high, height=0.75, color=colors, edgecolor='none')

xmax = max(high) * 1.11
xmin = min(low) * 2.2
offset = (xmax - xmin) * 0.01

ax.text(pct_cooling_min - offset, 4, f'{pct_cooling_min:+.1f}'.replace('-', '\u2212'), ha='right', va='center', fontsize=14)
ax.text(pct_cooling_max + offset, 4, f'{pct_cooling_max:+.1f}', ha='left', va='center', fontsize=14)
ax.text(pct_thermal_min - offset, 3, f'{pct_thermal_min:+.1f}'.replace('-', '\u2212'), ha='right', va='center', fontsize=14)
ax.text(pct_thermal_max + offset, 3, f'{pct_thermal_max:+.1f}', ha='left', va='center', fontsize=14)
ax.text(pct_model + offset, 2, f'{pct_model:+.1f}', ha='left', va='center', fontsize=14)
ax.text(pct_serving + offset, 1, f'{pct_serving:+.1f}', ha='left', va='center', fontsize=14)
ax.text(pct_hardware + offset, 0, f'{pct_hardware:+.1f}', ha='left', va='center', fontsize=14)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlim(xmin, xmax)
ax.set_ylim(-0.6, 4.6)
ax.set_yticks([])
ax.set_xlabel('Change in CET relative to baseline (%)', fontsize=16)
ax.tick_params(axis='x', labelsize=14)

handles = [Patch(facecolor='#2E75A0', edgecolor='none'),
           Patch(facecolor='#EBB320', edgecolor='none'),
           Patch(facecolor='none', edgecolor='none'),
           Patch(facecolor='#82B85A', edgecolor='none'),
           Patch(facecolor='#4BAFAF', edgecolor='none'),
           Patch(facecolor='#A04A7A', edgecolor='none')]
labels = ['Cooling type uncertainty',
          'Thermodynamic model uncertainty',
          '',
          'Line-of-sight improvement: model',
          'Line-of-sight improvement: serving',
          'Line-of-sight improvement: hardware']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.90),
           ncol=2, frameon=False, fontsize=15,
           handlelength=1.1, handleheight=1.1, columnspacing=2.0)

plt.savefig(str(RESULTS_ROOT / 'Figure/sensitivity_green_token_2025.png'), dpi=300, bbox_inches='tight')
plt.show()